## 드라이브 연결 및 설정

In [ ]:
!pip install kiwipiepy
!pip install transformers torch pandas

from google.colab import drive
drive.mount('/content/drive')


## corpus

In [ ]:
from pathlib import Path
import re
import html
import unicodedata
import pandas as pd


# RAW_XML_DIR = Path("data/dart/raw_xml")
# COMPANY_MASTER_PATH = Path("data/company_master.csv")

RAW_XML_DIR = Path("/content/drive/MyDrive/raw_xml")
COMPANY_MASTER_PATH = Path("/content/drive/MyDrive/company_master.csv")

TARGET_TITLE_REGEX = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}

TITLE_RE = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
MAIN_TITLE_RE = re.compile(
    r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\."
)


def clean_xml_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_report_text(text: str) -> str:
    text = clean_xml_text(text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_xml_filename(path: Path) -> tuple[str, int, str]:
    match = re.match(r"(\d{6})_(\d{4})_(\d+)\.xml$", path.name)
    if not match:
        raise ValueError(f"Unexpected XML filename: {path.name}")

    stock_code, fiscal_year, rcept_no = match.groups()
    return stock_code, int(fiscal_year), rcept_no


def extract_target_sections(xml_text: str) -> list[dict]:
    titles = []

    for match in TITLE_RE.finditer(xml_text):
        title = clean_xml_text(match.group(1))
        if MAIN_TITLE_RE.match(title):
            titles.append((title, match.start()))

    sections = []

    for i, (title, start) in enumerate(titles):
        section_name = None

        for name, pattern in TARGET_TITLE_REGEX.items():
            if re.search(pattern, title):
                section_name = name
                break

        if section_name is None:
            continue

        end = titles[i + 1][1] if i + 1 < len(titles) else len(xml_text)
        section_raw = xml_text[start:end]

        sections.append({
            "section": section_name,
            "text": clean_xml_text(section_raw),
        })

    return sections


def load_company_names(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=["stock_code", "company_name"])

    company_df = pd.read_csv(path, dtype={"stock_code": "string"})
    company_df["stock_code"] = (
        company_df["stock_code"]
        .astype("string")
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(6)
    )

    if "company_name" not in company_df.columns:
        return pd.DataFrame(columns=["stock_code", "company_name"])

    return (
        company_df[["stock_code", "company_name"]]
        .dropna()
        .drop_duplicates("stock_code")
    )


def build_firm_year_corpus(raw_xml_dir: Path) -> pd.DataFrame:
    rows = []

    xml_files = sorted(raw_xml_dir.glob("*.xml"))
    if not xml_files:
        raise FileNotFoundError(f"No XML files found in {raw_xml_dir}")

    for xml_path in xml_files:
        stock_code, fiscal_year, rcept_no = parse_xml_filename(xml_path)
        xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
        sections = extract_target_sections(xml_text)

        document = " ".join(section["text"] for section in sections)
        document_norm = normalize_report_text(document)

        rows.append({
            "stock_code": stock_code,
            "fiscal_year": fiscal_year,
            "rcept_no": rcept_no,
            "file_name": xml_path.name,
            "document": document,
            "document_norm": document_norm,
            "section_count": len({section["section"] for section in sections}),
            "total_word_count": len(document_norm.split()),
            "total_char_count": len(document_norm),
            "esg_year": fiscal_year + 1,
        })

    corpus_df = pd.DataFrame(rows)
    company_df = load_company_names(COMPANY_MASTER_PATH)

    corpus_df = corpus_df.merge(company_df, on="stock_code", how="left")

    ordered_cols = [
        "stock_code",
        "company_name",
        "fiscal_year",
        "rcept_no",
        "file_name",
        "document",
        "document_norm",
        "section_count",
        "total_word_count",
        "total_char_count",
        "esg_year",
    ]

    return (
        corpus_df[ordered_cols]
        .sort_values(["stock_code", "fiscal_year", "rcept_no"])
        .reset_index(drop=True)
    )


corpus_df = build_firm_year_corpus(RAW_XML_DIR)

print("rows:", len(corpus_df))
print("section_count distribution:")
print(corpus_df["section_count"].value_counts().sort_index())
corpus_df.head()

## EDA

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

GRADE_ORDER = ["D", "C", "B", "B+", "A", "A+", "S"]
GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}

eda_df = corpus_df.copy()

if "company_master" not in globals():
    company_master_path = Path("/content/drive/MyDrive/company_master.csv")
    company_master = pd.read_csv(company_master_path, dtype={"stock_code": "string"}, encoding="utf-8-sig")

for df in [eda_df, company_master]:
    df["stock_code"] = df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
    df["fiscal_year"] = pd.to_numeric(df["fiscal_year"], errors="coerce").astype("Int64")

needed_master_cols = [
    col for col in [
        "stock_code", "fiscal_year", "industry",
        "esg_grade", "e_grade", "s_grade", "g_grade"
    ]
    if col in company_master.columns
]

merge_cols = [
    col for col in ["industry", "esg_grade", "e_grade", "s_grade", "g_grade"]
    if col not in eda_df.columns and col in company_master.columns
]

if merge_cols:
    eda_df = eda_df.merge(
        company_master[["stock_code", "fiscal_year"] + merge_cols]
        .drop_duplicates(["stock_code", "fiscal_year"]),
        on=["stock_code", "fiscal_year"],
        how="left",
    )

if "esg_grade_num" not in eda_df.columns and "esg_grade" in eda_df.columns:
    eda_df["esg_grade_num"] = eda_df["esg_grade"].map(GRADE_MAP)

print("1. Overall corpus summary")
overall_summary = pd.DataFrame([{
    "firm_years": len(eda_df),
    "firms": eda_df["stock_code"].nunique(),
    "fiscal_years": eda_df["fiscal_year"].nunique(),
    "industries": eda_df["industry"].nunique() if "industry" in eda_df.columns else np.nan,
    "missing_esg_grade": eda_df["esg_grade"].isna().sum() if "esg_grade" in eda_df.columns else np.nan,
}])
display(overall_summary)

print("2. Yearly sample distribution")
year_sample = (
    eda_df
    .groupby("fiscal_year")
    .agg(
        firm_years=("stock_code", "size"),
        firms=("stock_code", "nunique"),
    )
    .reset_index()
)
display(year_sample)

if "industry" in eda_df.columns:
    print("3. Industry distribution")
    industry_sample = (
        eda_df
        .groupby("industry", dropna=False)
        .agg(
            firm_years=("stock_code", "size"),
            firms=("stock_code", "nunique"),
        )
        .assign(share=lambda df: df["firm_years"] / df["firm_years"].sum())
        .sort_values("firm_years", ascending=False)
        .reset_index()
    )
    display(industry_sample)

if "esg_grade_num" in eda_df.columns:
    print("4. Yearly ESG grade statistics")
    year_grade_stats = (
        eda_df
        .groupby("fiscal_year")["esg_grade_num"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )
    display(year_grade_stats)

if "esg_grade" in eda_df.columns:
    print("5. Yearly ESG grade distribution")
    year_grade_dist = (
        eda_df
        .pivot_table(
            index="fiscal_year",
            columns="esg_grade",
            values="stock_code",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=GRADE_ORDER, fill_value=0)
    )
    display(year_grade_dist)

    print("6. Overall ESG grade distribution")
    overall_grade_dist = (
        eda_df["esg_grade"]
        .value_counts(dropna=False)
        .reindex(GRADE_ORDER)
        .rename_axis("esg_grade")
        .reset_index(name="firm_years")
    )
    overall_grade_dist["share"] = overall_grade_dist["firm_years"] / overall_grade_dist["firm_years"].sum()
    display(overall_grade_dist)

if "industry" in eda_df.columns and "esg_grade_num" in eda_df.columns:
    print("7. Industry-level ESG grade statistics")
    industry_grade_stats = (
        eda_df
        .groupby("industry", dropna=False)["esg_grade_num"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("count", ascending=False)
        .reset_index()
    )
    display(industry_grade_stats)

length_cols = [col for col in ["total_word_count", "total_char_count", "kiwi_term_count"] if col in eda_df.columns]
if length_cols:
    print("8. Document length statistics")
    length_stats = (
        eda_df[length_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )
    display(length_stats)

    print("9. Yearly document length statistics")
    yearly_length_stats = (
        eda_df
        .groupby("fiscal_year")[length_cols]
        .agg(["count", "mean", "median", "std", "min", "max"])
    )
    display(yearly_length_stats)

## 증분설명력 분석

| 방법 | 선택값 | 점수 방식 | 추가 단어 수 | Base R² | Full R² | ΔR² | Full AIC | ΔAIC | Full BIC | ΔBIC | Partial F p-value | HC3 added p-value |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| fastText | θ = 0.70 | count | 3 | 0.3005 | 0.3283 | 0.0278 | 1305.8820 | -13.4499 | 1321.6532 | -9.5071 | 0.000093 | 1.43e-08 |
| fastText | θ = 0.70 | tfidf | 3 | 0.2789 | 0.3062 | 0.0274 | 1318.1637 | -12.7551 | 1333.9349 | -8.8123 | 0.000134 | 1.60e-07 |
| Kiwi embedding | threshold = 0.80 | count | 117 | 0.3005 | 0.4319 | 0.1314 | 1242.0472 | -77.2741 | 1257.8184 | -73.3313 | 8.68e-19 | 1.33e-18 |
| Kiwi embedding | threshold = 0.80 | tfidf | 117 | 0.2789 | 0.3985 | 0.1196 | 1263.8282 | -67.0658 | 1279.5994 | -63.1230 | 1.44e-16 | 2.25e-17 |

In [ ]:
import subprocess
import sys
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")

MODEL_NAME = "dragonkue/BGE-m3-ko"
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
BATCH_SIZE = 64 if DEVICE == "cuda" else 16

MIN_TERM_FREQ = 3
MIN_DOC_FREQ = 2
MAX_CANDIDATES = 30000
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90, 1.00]

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}

def normalize_term(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def token_key(term):
    return normalize_term(term).replace(" ", "_")


def threshold_label(theta):
    return f"{float(theta):.2f}".replace(".", "_")


def is_good_term(term):
    term = normalize_term(term)
    if len(term) < 2 or term.isdigit():
        return False
    if re.fullmatch(r"[0-9.,%/?????]+", term):
        return False
    if re.search(r"[\u3131-\u318E]", term):
        return False
    return True


def split_seed_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if is_good_term(term) and term.lower() != "nan" and term not in seen:
            terms.append(term)
            seen.add(term)
    return terms


def build_seed_query_df(seed_df):
    rows = []
    for idx, row in seed_df.reset_index(drop=True).iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue

        seed_term = normalize_term(row.get("seed_term", ""))
        for query_term in split_seed_terms(row):
            rows.append({
                "seed_id": idx,
                "dimension": dimension,
                "seed_term": seed_term,
                "query_term": query_term,
                "query_text": query_term,
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["dimension", "seed_term", "query_term"])
        .reset_index(drop=True)
    )


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_query_df = build_seed_query_df(seed_source_df)
seed_terms = set(seed_query_df["query_term"])

print("Seed query terms")
display(seed_query_df.groupby("dimension").size().rename("query_terms").reset_index())

kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text):
    candidates = []
    current = []

    for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
        form = normalize_term(token.form)
        if token.tag in NOUN_TAGS and is_good_term(form):
            candidates.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                phrase = normalize_term(" ".join(current))
                if is_good_term(phrase):
                    candidates.append(phrase)
                for i in range(len(current) - 1):
                    bigram = normalize_term(" ".join(current[i:i + 2]))
                    if is_good_term(bigram):
                        candidates.append(bigram)
            current = []

    if len(current) >= 2:
        phrase = normalize_term(" ".join(current))
        if is_good_term(phrase):
            candidates.append(phrase)
        for i in range(len(current) - 1):
            bigram = normalize_term(" ".join(current[i:i + 2]))
            if is_good_term(bigram):
                candidates.append(bigram)

    return candidates


appendix_embedding_doc_df = corpus_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "document_norm",
]].copy()
appendix_embedding_doc_df["kiwi_terms"] = appendix_embedding_doc_df["document_norm"].apply(kiwi_noun_candidates)
appendix_embedding_doc_df["kiwi_document"] = appendix_embedding_doc_df["kiwi_terms"].apply(
    lambda terms: " ".join(token_key(term) for term in terms)
)
appendix_embedding_doc_df["kiwi_term_count"] = appendix_embedding_doc_df["kiwi_terms"].apply(len)

term_counter = Counter()
doc_counter = Counter()
for terms in appendix_embedding_doc_df["kiwi_terms"]:
    term_counter.update(terms)
    doc_counter.update(set(terms))

candidate_rows = []
for term, freq in term_counter.most_common():
    if term in seed_terms:
        continue
    doc_freq = doc_counter[term]
    if freq < MIN_TERM_FREQ or doc_freq < MIN_DOC_FREQ:
        continue
    candidate_rows.append({
        "candidate_term": term,
        "term_frequency": freq,
        "doc_frequency": doc_freq,
    })

candidate_df = pd.DataFrame(candidate_rows).head(MAX_CANDIDATES).reset_index(drop=True)
if candidate_df.empty:
    raise ValueError("No candidate terms left. Lower MIN_TERM_FREQ/MIN_DOC_FREQ or inspect Kiwi extraction.")

print("candidate terms:", len(candidate_df))
display(candidate_df.head(20))


def encode_texts(model, texts, batch_size):
    embeddings = model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(embeddings, dtype=np.float32)


model = SentenceTransformer(MODEL_NAME, device=DEVICE)
seed_embeddings = encode_texts(model, seed_query_df["query_text"], BATCH_SIZE)
candidate_embeddings = encode_texts(model, candidate_df["candidate_term"], BATCH_SIZE)
similarity_matrix = seed_embeddings @ candidate_embeddings.T

best_rows = []
for cand_i, cand in candidate_df.reset_index(drop=True).iterrows():
    sims = similarity_matrix[:, cand_i]
    best_i = int(np.argmax(sims))
    best_seed = seed_query_df.iloc[best_i]

    dim_scores = {
        dimension: float(np.max(sims[list(indices)]))
        for dimension, indices in seed_query_df.groupby("dimension").groups.items()
    }
    sorted_dim_scores = sorted(dim_scores.items(), key=lambda x: x[1], reverse=True)
    best_dim, best_dim_score = sorted_dim_scores[0]
    second_dim_score = sorted_dim_scores[1][1] if len(sorted_dim_scores) > 1 else np.nan

    best_rows.append({
        "candidate_term": cand["candidate_term"],
        "term_frequency": cand["term_frequency"],
        "doc_frequency": cand["doc_frequency"],
        "best_dimension": best_dim,
        "best_dimension_score": best_dim_score,
        "second_dimension_score": second_dim_score,
        "dimension_margin": best_dim_score - second_dim_score,
        "best_seed_term": best_seed["seed_term"],
        "best_query_term": best_seed["query_term"],
    })

scored_candidate_df = pd.DataFrame(best_rows).sort_values(
    ["best_dimension", "best_dimension_score", "dimension_margin", "candidate_term"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

print("Top embedding-expanded candidates")
display(scored_candidate_df.head(30))

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)
appendix_embedding_tfidf_matrix = vectorizer.fit_transform(appendix_embedding_doc_df["kiwi_document"])
appendix_embedding_vocab = vectorizer.vocabulary_


def prepare_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_embedding_vocab and term not in seen:
            prepared.append((term, appendix_embedding_vocab[key]))
            seen.add(term)
    return prepared


def add_embedding_scores(score_df, meta_rows, dictionary, threshold, dimension, terms, prefix):
    prepared_terms = prepare_terms(terms)
    term_set = {term for term, _ in prepared_terms}
    tfidf_cols = [col for _, col in prepared_terms]

    count_feature = f"{dimension}_{prefix}_count"
    tfidf_feature = f"{dimension}_{prefix}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_embedding_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_embedding_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    meta_rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "prefix": prefix,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


appendix_embedding_score_df = appendix_embedding_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
appendix_embedding_meta_rows = []

for dimension in ["E", "S", "G"]:
    seed_terms_dim = seed_query_df.loc[seed_query_df["dimension"].eq(dimension), "query_term"].tolist()
    add_embedding_scores(appendix_embedding_score_df, appendix_embedding_meta_rows, "kiwi_seed_only", np.nan, dimension, seed_terms_dim, "kiwi_seed")

for theta in THRESHOLDS:
    kept = scored_candidate_df[scored_candidate_df["best_dimension_score"].ge(theta)]
    prefix = f"kiwi_expanded_{threshold_label(theta)}"
    for dimension in ["E", "S", "G"]:
        terms = kept.loc[kept["best_dimension"].eq(dimension), "candidate_term"].tolist()
        add_embedding_scores(appendix_embedding_score_df, appendix_embedding_meta_rows, "kiwi_embedding_expanded", theta, dimension, terms, prefix)

appendix_embedding_meta_df = pd.DataFrame(appendix_embedding_meta_rows)
for prefix in appendix_embedding_meta_df["prefix"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        appendix_embedding_score_df[f"ESG_{prefix}_{score_type}"] = sum(
            appendix_embedding_score_df[f"{dimension}_{prefix}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

company_master = company_master.copy()
company_master["stock_code"] = company_master["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
appendix_embedding_score_df["stock_code"] = appendix_embedding_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    appendix_embedding_score_df[col] = pd.to_numeric(appendix_embedding_score_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
appendix_embedding_analysis_df = appendix_embedding_score_df.merge(
    company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

print("Kiwi embedding expanded term counts by threshold")
display(
    appendix_embedding_meta_df[appendix_embedding_meta_df["dictionary"].eq("kiwi_embedding_expanded")]
    .pivot_table(index="threshold", columns="dimension", values="term_count", aggfunc="first")
)

# Incremental explanatory power: does Kiwi embedding expanded-only score improve over seed?
import statsmodels.api as sm


def appendix_embedding_zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def appendix_embedding_fit_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        return None, None, reg_df
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(appendix_embedding_zscore)
    X = sm.add_constant(X, has_constant="add")
    plain_model = sm.OLS(y, X).fit()
    hc3_model = sm.OLS(y, X).fit(cov_type="HC3")
    return plain_model, hc3_model, reg_df


embedding_incremental_rows = []
seed_prefix = "kiwi_seed"
for _, group in appendix_embedding_meta_df[appendix_embedding_meta_df["dictionary"].eq("kiwi_embedding_expanded")].groupby(["threshold", "prefix"], dropna=False):
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    added_term_count = int(group["term_count"].sum())
    added_dictionary_term_count = int(group["dictionary_term_count"].sum())

    for score_type in ["count", "tfidf"]:
        seed_feature = f"ESG_{seed_prefix}_{score_type}"
        added_feature = f"ESG_{prefix}_{score_type}"
        if seed_feature not in appendix_embedding_analysis_df.columns or added_feature not in appendix_embedding_analysis_df.columns:
            continue

        # Kiwi embedding expansion candidates exclude seed terms, so the expanded feature is already added-only.
        base_plain, base_hc3, base_df = appendix_embedding_fit_ols(
            appendix_embedding_analysis_df,
            "esg_grade_num",
            [seed_feature, "total_word_count"],
        )
        full_plain, full_hc3, full_df = appendix_embedding_fit_ols(
            appendix_embedding_analysis_df,
            "esg_grade_num",
            [seed_feature, added_feature, "total_word_count"],
        )
        if base_plain is None or full_plain is None:
            continue

        f_stat, f_pvalue, df_diff = full_plain.compare_f_test(base_plain)
        embedding_incremental_rows.append({
            "dictionary": "kiwi_embedding_expanded",
            "threshold": threshold,
            "score_type": score_type,
            "seed_feature": seed_feature,
            "added_feature": added_feature,
            "n": int(full_plain.nobs),
            "added_term_count": added_term_count,
            "added_dictionary_term_count": added_dictionary_term_count,
            "base_r2": float(base_plain.rsquared),
            "full_r2": float(full_plain.rsquared),
            "delta_r2": float(full_plain.rsquared - base_plain.rsquared),
            "base_adj_r2": float(base_plain.rsquared_adj),
            "full_adj_r2": float(full_plain.rsquared_adj),
            "delta_adj_r2": float(full_plain.rsquared_adj - base_plain.rsquared_adj),
            "base_aic": float(base_plain.aic),
            "full_aic": float(full_plain.aic),
            "delta_aic_full_minus_base": float(full_plain.aic - base_plain.aic),
            "base_bic": float(base_plain.bic),
            "full_bic": float(full_plain.bic),
            "delta_bic_full_minus_base": float(full_plain.bic - base_plain.bic),
            "partial_f": float(f_stat),
            "partial_f_p_value": float(f_pvalue),
            "df_diff": float(df_diff),
            "added_coef_hc3": float(full_hc3.params[added_feature]),
            "added_p_value_hc3": float(full_hc3.pvalues[added_feature]),
        })

appendix_embedding_incremental_ols_df = (
    pd.DataFrame(embedding_incremental_rows)
    .sort_values(["score_type", "delta_adj_r2"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Kiwi embedding added-only incremental OLS over seed baseline")
display(appendix_embedding_incremental_ols_df)

if not appendix_embedding_incremental_ols_df.empty:
    print("Best Kiwi embedding threshold by adjusted R2 gain")
    display(
        appendix_embedding_incremental_ols_df
        .sort_values("delta_adj_r2", ascending=False)
        .head(10)
    )



## 상관분석

In [ ]:
from scipy.stats import spearmanr

FINAL_EXPANDED_DICTIONARY_PATH = Path("/content/drive/MyDrive/candidate_kiwi_embedding_threshold_0_80_dictionary.csv")

GRADE_BY_DIMENSION = {
    "E": "e_grade_num",
    "S": "s_grade_num",
    "G": "g_grade_num",
    "ESG": "esg_grade_num",
}


def load_expanded_dictionary(path):
    expanded_df = pd.read_csv(path, encoding="utf-8-sig")
    term_col = next(
        col for col in ["candidate_term", "query_term", "seed_term", "term"]
        if col in expanded_df.columns
    )
    loaded_df = expanded_df[["dimension", term_col]].rename(columns={term_col: "candidate_term"}).copy()
    loaded_df["dictionary"] = "final_expanded"
    loaded_df["candidate_term"] = loaded_df["candidate_term"].map(normalize_term)
    loaded_df = loaded_df[loaded_df["dimension"].isin(["E", "S", "G"])]
    loaded_df = loaded_df[loaded_df["candidate_term"].ne("")]
    return loaded_df.drop_duplicates(["dictionary", "dimension", "candidate_term"]).reset_index(drop=True)


def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)

    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


required_previous_vars = [
    "appendix_embedding_doc_df",
    "appendix_embedding_tfidf_matrix",
    "appendix_embedding_vocab",
    "company_master",
]
missing_vars = [name for name in required_previous_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(f"Run the ????? ?? cell first. Missing: {missing_vars}")

if not str(FINAL_EXPANDED_DICTIONARY_PATH).strip():
    raise ValueError("FINAL_EXPANDED_DICTIONARY_PATH must point to the selected threshold 0.80 expanded dictionary.")

correlation_dictionary_df = load_expanded_dictionary(FINAL_EXPANDED_DICTIONARY_PATH)

print("Dictionary term counts")
display(
    correlation_dictionary_df
    .groupby(["dictionary", "dimension"])
    .size()
    .rename("dictionary_term_count")
    .reset_index()
)


def prepare_final_dictionary_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_embedding_vocab and term not in seen:
            prepared.append((term, key, appendix_embedding_vocab[key]))
            seen.add(term)
    return prepared


def add_final_dimension_scores(score_df, feature_meta_rows, dictionary, dimension, terms):
    prepared_terms = prepare_final_dictionary_terms(terms)
    term_set = {term for term, _, _ in prepared_terms}
    tfidf_cols = [col for _, _, col in prepared_terms]

    count_feature = f"{dimension}_{dictionary}_count"
    tfidf_feature = f"{dimension}_{dictionary}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_embedding_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_embedding_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    feature_meta_rows.append({
        "dictionary": dictionary,
        "dimension": dimension,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


final_score_df = appendix_embedding_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
final_feature_meta_rows = []

for (dictionary, dimension), group in correlation_dictionary_df.groupby(["dictionary", "dimension"]):
    add_final_dimension_scores(final_score_df, final_feature_meta_rows, dictionary, dimension, group["candidate_term"].tolist())

for dictionary in correlation_dictionary_df["dictionary"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        final_score_df[f"ESG_{dictionary}_{score_type}"] = sum(
            final_score_df[f"{dimension}_{dictionary}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

final_feature_meta_df = pd.DataFrame(final_feature_meta_rows)
final_score_df["stock_code"] = final_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    final_score_df[col] = pd.to_numeric(final_score_df[col], errors="coerce").astype("Int64")

final_grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]

final_level_analysis_df = final_score_df.merge(
    company_master[final_grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

final_corr_rows = []
for dictionary in correlation_dictionary_df["dictionary"].drop_duplicates():
    dictionary_meta_df = final_feature_meta_df[final_feature_meta_df["dictionary"].eq(dictionary)]
    for dimension in ["E", "S", "G", "ESG"]:
        grade_col = GRADE_BY_DIMENSION[dimension]
        term_count = (
            int(dictionary_meta_df.loc[dictionary_meta_df["dimension"].eq(dimension), "term_count"].iloc[0])
            if dimension != "ESG" else int(dictionary_meta_df["term_count"].sum())
        )
        dictionary_term_count = (
            int(dictionary_meta_df.loc[dictionary_meta_df["dimension"].eq(dimension), "dictionary_term_count"].iloc[0])
            if dimension != "ESG" else int(dictionary_meta_df["dictionary_term_count"].sum())
        )

        for score_type in ["count", "tfidf"]:
            feature = f"{dimension}_{dictionary}_{score_type}"
            rho, pvalue, n = safe_spearman(final_level_analysis_df[feature], final_level_analysis_df[grade_col])
            final_corr_rows.append({
                "scope": "pooled",
                "dictionary": dictionary,
                "dimension": dimension,
                "score_type": score_type,
                "feature": feature,
                "grade_col": grade_col,
                "term_count": term_count,
                "dictionary_term_count": dictionary_term_count,
                "spearman_rho": rho,
                "p_value": pvalue,
                "n": n,
            })

final_correlation_df = pd.DataFrame(final_corr_rows)

final_year_corr_rows = []
for _, row in final_correlation_df.iterrows():
    for fiscal_year, year_df in final_level_analysis_df.groupby("fiscal_year"):
        rho, pvalue, n = safe_spearman(year_df[row["feature"]], year_df[row["grade_col"]])
        final_year_corr_rows.append({
            "scope": "yearly",
            "fiscal_year": fiscal_year,
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": row["feature"],
            "grade_col": row["grade_col"],
            "term_count": row["term_count"],
            "dictionary_term_count": row["dictionary_term_count"],
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

final_year_correlation_df = pd.DataFrame(final_year_corr_rows)

print("Pooled Spearman: final expanded dictionary threshold 0.80")
display(final_correlation_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Pooled pivot")
display(final_correlation_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="spearman_rho",
    aggfunc="first",
))

print("Yearly Spearman: final expanded dictionary threshold 0.80")
display(final_year_correlation_df.sort_values(["fiscal_year", "dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Yearly pivot")
display(final_year_correlation_df.pivot_table(
    index=["dictionary", "score_type", "fiscal_year"],
    columns="dimension",
    values="spearman_rho",
    aggfunc="first",
))


In [ ]:
import subprocess
import sys

from scipy.stats import kruskal, mannwhitneyu

try:
    import scikit_posthocs as sp
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-posthocs"])
    import scikit_posthocs as sp

HIGH_GRADE_MIN = 4  # A, A+, S
LOW_GRADE_MAX = 3   # D, C, B, B+


def run_mannwhitney_feature_test(data, feature_meta, high_min=HIGH_GRADE_MIN, low_max=LOW_GRADE_MAX):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        high_values = tmp.loc[tmp[grade_col] >= high_min, feature]
        low_values = tmp.loc[tmp[grade_col] <= low_max, feature]

        statistic, pvalue = np.nan, np.nan
        if len(high_values) >= 2 and len(low_values) >= 2:
            try:
                statistic, pvalue = mannwhitneyu(high_values, low_values, alternative="two-sided")
            except ValueError:
                pass

        rank_biserial = (
            2 * statistic / (len(high_values) * len(low_values)) - 1
            if pd.notna(statistic) and len(high_values) > 0 and len(low_values) > 0
            else np.nan
        )

        rows.append({
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": feature,
            "grade_col": grade_col,
            "test": "Mann-Whitney U",
            "group_rule": "high=A 이상, low=B+ 이하",
            "n_high": len(high_values),
            "n_low": len(low_values),
            "high_mean": high_values.mean(),
            "low_mean": low_values.mean(),
            "mean_diff_high_minus_low": high_values.mean() - low_values.mean(),
            "high_median": high_values.median(),
            "low_median": low_values.median(),
            "mannwhitney_u": statistic,
            "rank_biserial": rank_biserial,
            "p_value": pvalue,
        })

    return pd.DataFrame(rows)


def run_kruskal_feature_test(data, feature_meta):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        grouped = [
            (grade, group[feature].dropna())
            for grade, group in tmp.groupby(grade_col)
        ]
        grouped = [(grade, values) for grade, values in grouped if len(values) > 0]
        groups = [values.values for _, values in grouped]

        statistic, pvalue = np.nan, np.nan
        if len(groups) >= 2 and sum(len(values) for values in groups) >= 3:
            try:
                statistic, pvalue = kruskal(*groups)
            except ValueError:
                pass

        rows.append({
            "dictionary": row["dictionary"],
            "dimension": row["dimension"],
            "score_type": row["score_type"],
            "feature": feature,
            "grade_col": grade_col,
            "test": "Kruskal-Wallis",
            "grade_groups": ", ".join(str(grade) for grade, _ in grouped),
            "grade_group_counts": ", ".join(f"{grade}:{len(values)}" for grade, values in grouped),
            "n_groups": len(groups),
            "n_total": sum(len(values) for values in groups),
            "kruskal_h": statistic,
            "p_value": pvalue,
        })

    return pd.DataFrame(rows)


def run_dunn_posthoc_test(data, feature_meta, p_adjust="holm"):
    rows = []
    for _, row in feature_meta.iterrows():
        feature = row["feature"]
        grade_col = row["grade_col"]
        tmp = (
            data[[feature, grade_col]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .copy()
        )

        grade_counts = tmp[grade_col].value_counts().sort_index()
        if len(grade_counts) < 2:
            continue

        try:
            posthoc = sp.posthoc_dunn(tmp, val_col=feature, group_col=grade_col, p_adjust=p_adjust)
        except ValueError:
            continue

        grades = list(posthoc.index)
        for i, grade_a in enumerate(grades):
            for grade_b in grades[i + 1:]:
                values_a = tmp.loc[tmp[grade_col].eq(grade_a), feature]
                values_b = tmp.loc[tmp[grade_col].eq(grade_b), feature]
                rows.append({
                    "dictionary": row["dictionary"],
                    "dimension": row["dimension"],
                    "score_type": row["score_type"],
                    "feature": feature,
                    "grade_col": grade_col,
                    "test": "Dunn posthoc",
                    "p_adjust": p_adjust,
                    "grade_a": grade_a,
                    "grade_b": grade_b,
                    "n_a": len(values_a),
                    "n_b": len(values_b),
                    "median_a": values_a.median(),
                    "median_b": values_b.median(),
                    "median_diff_a_minus_b": values_a.median() - values_b.median(),
                    "p_value_adj": posthoc.loc[grade_a, grade_b],
                })

    return pd.DataFrame(rows)


final_mannwhitney_df = run_mannwhitney_feature_test(final_level_analysis_df, final_correlation_df)
final_kruskal_df = run_kruskal_feature_test(final_level_analysis_df, final_correlation_df)
final_dunn_df = run_dunn_posthoc_test(final_level_analysis_df, final_correlation_df)

print("Mann-Whitney U test: high grade (A 이상) vs low grade (B+ 이하)")
display(final_mannwhitney_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Mann-Whitney pivot: p-value")
display(final_mannwhitney_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="p_value",
    aggfunc="first",
))

print("Kruskal-Wallis test: all observed grade groups")
display(final_kruskal_df.sort_values(["dictionary", "dimension", "score_type"]).reset_index(drop=True))

print("Kruskal-Wallis pivot: p-value")
display(final_kruskal_df.pivot_table(
    index=["dictionary", "score_type"],
    columns="dimension",
    values="p_value",
    aggfunc="first",
))

print("Dunn posthoc test after Kruskal-Wallis: Holm-adjusted p-value")
display(final_dunn_df.sort_values(["dictionary", "dimension", "score_type", "p_value_adj"]).reset_index(drop=True))

print("Dunn posthoc significant pairs: adjusted p < 0.05")
display(
    final_dunn_df[final_dunn_df["p_value_adj"] < 0.05]
    .sort_values(["dictionary", "dimension", "score_type", "p_value_adj"])
    .reset_index(drop=True)
)


## ESG 감성분석

주요 결과:

- 평균 ESG 감성 점수와 실제 ESG 등급 변화량 간에 약한 양의 상관관계가 있는 것으로 나타남.

- 이는 통계적으로 유의수준 10% 하에서 유의미한 수치임. 보고서 내 ESG 감성 점수가 높을수록 다음 해 ESG 등급이 상승하는 경향을 일부 보임.

In [ ]:
from transformers import pipeline

if "sentiment_pipe" not in globals():
    sentiment_pipe = pipeline("sentiment-analysis", model="snunlp/KR-FinBert-SC", tokenizer="snunlp/KR-FinBert-SC", truncation=True, max_length=256)

if "kiwi" not in globals():
    from kiwipiepy import Kiwi
    kiwi = Kiwi()

seed_words = set(pd.read_csv('/content/drive/MyDrive/candidate_kiwi_embedding_threshold_0_80_dictionary.csv')['seed_term'].dropna())

results = []

for idx, row in corpus_df.iterrows():
    text = str(row["document_norm"])
    if not text.strip(): continue

    sentences = kiwi.split_into_sents(text)
    total_words = 0

    s_pos_w, s_neg_w, s_pos_s, s_neg_s = 0, 0, 0, 0

    for sent in sentences:
        sent_text = sent.text
        tokens = [t.form for t in kiwi.tokenize(sent_text) if t.tag.startswith('N') or t.tag.startswith('V')]
        total_words += len(tokens)
        found_seed = [w for w in tokens if w in seed_words]

        if not found_seed:
            continue

        label = sentiment_pipe(sent_text)[0]['label']

        if label == 'positive' and found_seed:
            s_pos_w += len(found_seed); s_pos_s += 1
        elif label == 'negative' and found_seed:
            s_neg_w += len(found_seed); s_neg_s += 1

    base_info = {
        "stock_code": row["stock_code"],
        "company_name": row.get("company_name", ""),
        "fiscal_year": row["fiscal_year"],
        "esg_year": row["fiscal_year"] + 1,
        "esg_grade_num": row.get("esg_grade_num_real", row.get("esg_grade_num", None)),
        "esg_sentence_count": len(sentences),
        "total_word_count": total_words
    }

    results.append({
        **base_info,
        "dictionary_label": "seed",
        "mean_esg_sentiment": (s_pos_w - s_neg_w) / total_words if total_words > 0 else 0,
        "positive_esg_share": s_pos_w / total_words if total_words > 0 else 0,
        "negative_esg_share": s_neg_w / total_words if total_words > 0 else 0,
        "positive_esg_sentence_count": s_pos_s,
        "negative_esg_sentence_count": s_neg_s,
    })

v_2_sentiment_analysis_df = pd.DataFrame(results)


In [ ]:
v_2_sentiment_analysis_df.to_csv("v_2_sentiment_analysis_df.csv", index=False, encoding="utf-8-sig")


In [ ]:
esg_df = company_master.copy()

esg_df["stock_code"] = esg_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
v_2_sentiment_analysis_df["stock_code"] = (
    v_2_sentiment_analysis_df["stock_code"]
    .astype("string")
    .str.extract(r"(\d+)", expand=False)
    .str.zfill(6)
)

if "esg_grade_num" not in esg_df.columns:
    esg_df["esg_grade_num"] = esg_df["esg_grade"].map(GRADE_MAP)

v_2_sentiment_analysis_df = v_2_sentiment_analysis_df.drop(
    columns=["company_name", "esg_grade_num"],
    errors="ignore",
)

v_2_sentiment_analysis_df = pd.merge(
    v_2_sentiment_analysis_df,
    esg_df[["stock_code", "fiscal_year", "company_name", "esg_grade_num"]],
    on=["stock_code", "fiscal_year"],
    how="left",
)


In [ ]:
analysis_df = v_2_sentiment_analysis_df.copy()

required_cols = {
    "dictionary_label",
    "stock_code",
    "company_name",
    "fiscal_year",
    "esg_year",
    "esg_grade_num",
    "mean_esg_sentiment",
    "positive_esg_share",
    "negative_esg_share",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "esg_sentence_count",
    "total_word_count",
}

missing = sorted(required_cols - set(analysis_df.columns))
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

use_cols = [
    "dictionary_label", "stock_code", "company_name", "fiscal_year", "esg_year", "esg_grade_num",
    "mean_esg_sentiment", "positive_esg_share", "negative_esg_share",
    "positive_esg_sentence_count", "negative_esg_sentence_count",
    "esg_sentence_count", "total_word_count",
]

panel_df = analysis_df[use_cols].copy()
for col in ["fiscal_year", "esg_year", "esg_grade_num"]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")
for col in [c for c in use_cols if c not in {"dictionary_label", "stock_code", "company_name"}]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")

panel_df = panel_df.sort_values(["dictionary_label", "stock_code", "fiscal_year"]).reset_index(drop=True)

print("panel rows:", len(panel_df))
print("firm-year rows per dictionary:")
display(panel_df.groupby("dictionary_label").agg(rows=("stock_code", "size"), firms=("stock_code", "nunique")))
display(panel_df.head())

In [ ]:
year_summary = (
    panel_df.groupby(["dictionary_label", "fiscal_year"])
    .agg(
        rows=("stock_code", "size"),
        firms=("stock_code", "nunique"),
        mean_grade=("esg_grade_num", "mean"),
        mean_sentiment=("mean_esg_sentiment", "mean"),
    )
    .reset_index()
)

years_per_firm = (
    panel_df.groupby(["dictionary_label", "stock_code"])["fiscal_year"]
    .nunique()
    .reset_index(name="observed_years")
    .groupby(["dictionary_label", "observed_years"])
    .size()
    .reset_index(name="firm_count")
)

print("연도별 요약")
display(year_summary)
print("기업별 관측연도 수")
display(years_per_firm)

In [ ]:
change_base_cols = [
    "mean_esg_sentiment",
    "positive_esg_share",
    "negative_esg_share",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "esg_sentence_count",
    "total_word_count",
    "esg_grade_num",
]

change_df = panel_df.copy()
group_keys = ["dictionary_label", "stock_code"]

for col in change_base_cols:
    change_df[f"lag_{col}"] = change_df.groupby(group_keys)[col].shift(1)
    change_df[f"delta_{col}"] = change_df[col] - change_df[f"lag_{col}"]

change_df["year_gap"] = change_df["fiscal_year"] - change_df.groupby(group_keys)["fiscal_year"].shift(1)
change_df = change_df[change_df["year_gap"] == 1].copy()

change_df["grade_change_group"] = np.select(
    [change_df["delta_esg_grade_num"] > 0, change_df["delta_esg_grade_num"] < 0],
    ["등급 상승", "등급 하락"],
    default="등급 유지",
)

print("change rows:", len(change_df))
display(
    change_df[[
        "dictionary_label", "stock_code", "company_name", "fiscal_year",
        "esg_grade_num", "lag_esg_grade_num", "delta_esg_grade_num",
        "mean_esg_sentiment", "lag_mean_esg_sentiment",
        "delta_mean_esg_sentiment", "grade_change_group",
    ]].head(20)
)


In [ ]:
change_summary = (
    change_df.groupby(["dictionary_label", "fiscal_year", "grade_change_group"])
    .size()
    .rename("row_count")
    .reset_index()
)

overall_change_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])
    .size()
    .rename("row_count")
    .reset_index()
)

print("전체 등급 변화 그룹 분포")
display(overall_change_summary)
print("연도별 등급 변화 그룹 분포")
display(change_summary)

In [ ]:
text_all_cols = [
    "mean_esg_sentiment", "delta_mean_esg_sentiment",
    "positive_esg_share", "delta_positive_esg_share",
    "negative_esg_share", "delta_negative_esg_share",
    "positive_esg_sentence_count", "delta_positive_esg_sentence_count",
    "negative_esg_sentence_count", "delta_negative_esg_sentence_count",
    "esg_sentence_count", "delta_esg_sentence_count",
    "total_word_count", "delta_total_word_count"
]

def spearman_delta_table(data, y_col, x_cols):
    rows = []
    for dictionary_label, group in data.groupby("dictionary_label"):
        for x_col in x_cols:
            tmp = group[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 3 or tmp[y_col].nunique() < 2 or tmp[x_col].nunique() < 2:
                rho, pvalue = np.nan, np.nan
            else:
                rho, pvalue = spearmanr(tmp[y_col], tmp[x_col])
            rows.append({
                "dictionary_label": dictionary_label,
                "feature": x_col,
                "target": y_col,
                "n": len(tmp),
                "spearman_rho": rho,
                "p_value": pvalue,
            })
    return pd.DataFrame(rows).sort_values(["feature", "spearman_rho"], ascending=[True, False])

spearman_change_df = spearman_delta_table(change_df, "delta_esg_grade_num", text_all_cols)
display(spearman_change_df)



In [ ]:
group_cols = [
    "mean_esg_sentiment", "delta_mean_esg_sentiment",
    "positive_esg_share", "delta_positive_esg_share",
    "negative_esg_share", "delta_negative_esg_share",
    "esg_sentence_count", "delta_esg_sentence_count",
    "total_word_count", "delta_total_word_count"
]

group_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])[group_cols]
    .agg(["count", "mean", "median", "std"])
)
display(group_summary)

In [ ]:
from scipy.stats import kruskal

test_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    for col in group_cols:
        groups = [
            g[col].replace([np.inf, -np.inf], np.nan).dropna().values
            for _, g in dictionary_group.groupby("grade_change_group")
        ]
        groups = [g for g in groups if len(g) > 0]
        stat, pvalue = np.nan, np.nan
        if len(groups) >= 2 and sum(len(g) for g in groups) >= 3:
            try:
                stat, pvalue = kruskal(*groups)
            except ValueError:
                pass
        test_rows.append({
            "dictionary_label": dictionary_label,
            "feature": col,
            "test": "Kruskal-Wallis",
            "statistic": stat,
            "p_value": pvalue,
        })

kruskal_df = pd.DataFrame(test_rows)
display(kruskal_df)

In [ ]:
def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std

def robust_delta_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        raise ValueError(f"too few complete rows after dropna: n={len(reg_df)}, predictors={len(x_cols)}")
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    result = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_HC3": model.bse.values,
        "t": model.tvalues.values,
        "p_value": model.pvalues.values,
    })
    return model, result, reg_df

model_specs = {
    "M1_sentiment_only_level": ["mean_esg_sentiment"],
    "M1_sentiment_only_delta": ["delta_mean_esg_sentiment"],
    "M2_sentiment_plus_volume_level": ["mean_esg_sentiment", "total_word_count"],
    "M2_sentiment_plus_volume_delta": ["delta_mean_esg_sentiment", "delta_total_word_count"],
    "M3_pos_neg_plus_volume_level": ["positive_esg_share", "negative_esg_share", "total_word_count"],
    "M3_pos_neg_plus_volume_delta": ["delta_positive_esg_share", "delta_negative_esg_share", "delta_total_word_count"],
}

ols_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    print("\n" + "=" * 80)
    print("dictionary_label:", dictionary_label)
    for name, x_cols in model_specs.items():
        try:
            model, result, reg_df = robust_delta_ols(dictionary_group, "delta_esg_grade_num", x_cols)
            print("\n" + name, "n=", len(reg_df), "R2=", round(model.rsquared, 4))
            display(result)
            for _, row in result.iterrows():
                ols_rows.append({
                    "dictionary_label": dictionary_label,
                    "model": name,
                    "n": len(reg_df),
                    "r2": model.rsquared,
                    **row.to_dict(),
                })
        except Exception as exc:
            print(f"{name} Error: {exc}")


In [ ]:
if not spearman_change_df.empty:
    valid_df = (
        spearman_change_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["spearman_rho"])
    )

    if not valid_df.empty:
        top = (
            valid_df
            .assign(abs_rho=lambda df: df["spearman_rho"].abs())
            .sort_values("abs_rho", ascending=False)
            .iloc[0]
        )
        print("상관관계 분석 요약:")
        print(
            f"{top['dictionary_label']} 사전 기준에서 {top['feature']}와 "
            f"delta_esg_grade_num의 Spearman rho={top['spearman_rho']:.3f}, "
            f"p={top['p_value']:.3f}입니다."
        )

## 부록

#### 생존분석 (미정)

결과에 대한 설명: p값 0.095
표본 수 70행(이벤트 11건)
표본 수가 적어서 실질적인 효과가 존재하더라도 유의확률이 높게 나옴(검정력 부족)

순수 빈도 지수는 상투적인 어미가 존재하기 때문에 문맥을 정확히 타격하지 못했다고 보여짐

위험계수는 0.0402 위험비는 1.0410
ESG 통합 지수가 1단위 증가할수록(사업보고서에 ESG 관련 긍정적인 말들을 더 많이 적을수록) 차년도 ESG 등급이 하락할 위험이 오히려 4.1%씩 증가한다는 모순적인 결과가 나옴

이러한 이유로 인해 부록에 추가함 감성분석 코드 수정 시 결과가 바뀔 수 있음!

확장 사전 수정 이후: p값과 위험계수와 위험비 전부 차이가 없어 생존분석은 해당 프로젝트에서 유의미한 값이나 변화량을 보여주지 못하여 우리가 확인하려는 부분과 먼 연관성을 보임.

In [ ]:
!pip install transformers tqdm lifelines

In [ ]:
#생존분석 필수 라이브러리 로드 및 가설 설정
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

print("⏳ [생존분석 1단계] 감성분석 결과 기반 생존분석 데이터셋 준비...")

df_survival_base = v_2_sentiment_analysis_df.copy()

print(f"✅ 생존분석 기본 데이터 확보 완료: 총 {len(df_survival_base)}행 구조 확인")

In [ ]:
# [감성지수 산출 로직 정상화]
print("⏳ [생존분석 2단계] 비율(Share)을 배제하고 순수 문장 개수(Count) 기반으로 지수 재연산 중...")

# 1. 꼬임 방지를 위해 순수 Count 컬럼만 명확하게 지정
pos_count_col = 'positive_esg_sentence_count'
neg_count_col = 'negative_esg_sentence_count'

# 2. 안전하게 긍정 개수에서 부정 개수를 차감하여 지수 생성 (정수형 스케일 확보)
df_survival_base['Total_ESG_idx'] = df_survival_base[pos_count_col] - df_survival_base[neg_count_col]

print("✅ 통합 ESG Net Sentiment 지수 정상화 완료 (Total_ESG_idx)")
print(df_survival_base[['stock_code', 'fiscal_year', pos_count_col, neg_count_col, 'Total_ESG_idx']].head(5))

In [ ]:
# [중복 연도 제거 및 이벤트 전면 재추적]
print("⏳ [생존분석 3단계] 중복 데이터 정제 및 시계열 생존 피처(Event) 정밀 복구 중...")

# 1. 데이터프레임 내 동일 기업-동일 연도 중복 행 제거 (평균값 또는 첫 행 기준으로 단일화)
# 여기서는 데이터의 정성적 가치를 보존하기 위해 중복된 행 중 첫 번째 행만 남깁니다.
df_cleaned = df_survival_base.drop_duplicates(subset=['stock_code', 'fiscal_year'], keep='first').copy()
print(f"💡 중복 행 정제 완료: 기존 {len(df_survival_base)}행 ➡️ 정제 후 {len(df_cleaned)}행 데이터 확보")

# 2. 기업별, 연도별 시계열 엄격 정렬
df_surv = df_cleaned.sort_values(['stock_code', 'fiscal_year']).reset_index(drop=True)

# 3. 수치형 등급 피처 고정
grade_col = 'esg_grade_num'
df_surv['grade_idx_numeric'] = pd.to_numeric(df_surv[grade_col], errors='coerce')
df_surv['grade_idx_numeric'] = df_surv.groupby('stock_code')['grade_idx_numeric'].ffill()

# 4. 관측 기간(Duration) 계산
df_surv['min_year'] = df_surv.groupby('stock_code')['fiscal_year'].transform('min')
df_surv['duration'] = df_surv['fiscal_year'] - df_surv['min_year'] + 1

# 5. 시계열 무결성이 확보된 상태에서 등급 하락 이벤트(Event) 추적
df_surv['prev_grade_num'] = df_surv.groupby('stock_code')['grade_idx_numeric'].shift(1)
df_surv['event_shift'] = np.where(df_surv['grade_idx_numeric'] > df_surv['prev_grade_num'], 1, 0)

df_surv['first_grade_num'] = df_surv.groupby('stock_code')['grade_idx_numeric'].transform('first')
df_surv['event_baseline'] = np.where(df_surv['grade_idx_numeric'] > df_surv['first_grade_num'], 1, 0)

# 최종 이벤트 결합
df_surv['event'] = np.where((df_surv['event_shift'] == 1) | (df_surv['event_baseline'] == 1), 1, 0)
df_surv['event'] = df_surv['event'].fillna(0).astype(int)

print(f"🎯 피처 엔지니어링 최종 완료 ➡️ 정정된 총 등급 하락(사망 사건) 횟수: {df_surv['event'].sum()}건 발생")

In [ ]:
# [Cox PH 모델 최종 핏팅]
print("⏳ [생존분석 4단계] 정제된 70행 표본 및 11건의 순수 이벤트를 기반으로 Cox PH 모형 적합 중...")

# 분석 필수 변수만 추출 및 결측치 제거
keep_cols = ['duration', 'event', 'Total_ESG_idx']
df_model = df_surv[keep_cols].dropna()

# 콕스 비례위험모형 선언 및 학습
cph = CoxPHFitter()
cph.fit(df_model, duration_col='duration', event_col='event')

print("\n🏆 [최종 통계 결과] 미확장 Baseline 통합 ESG 지수 기반 Cox 생존분석 요약 리포트")
print("=" * 80)
cph.print_summary()
print("=" * 80)

In [ ]:
# [최종 지표 정형화 출력]
print("⏳ [생존분석 5단계] 보고서 삽입용 최종 핵심 수치 정형화 추출 중...")

summary_df = cph.summary
coef_val = summary_df.loc['Total_ESG_idx', 'coef']
exp_coef_val = summary_df.loc['Total_ESG_idx', 'exp(coef)']
p_val = summary_df.loc['Total_ESG_idx', 'p']
concordance_val = cph.concordance_index_
aic_val = cph.AIC_partial_

print("\n📝 [최종 보고서 본문/결론 섹션 삽입용 텍스트 요약]")
print("-" * 65)
print(f"   - 위험 계수 (Coefficient): {coef_val:.4f}")
print(f"   - 위험비 (Hazard Ratio, exp(coef)): {exp_coef_val:.4f}")
print(f"   - 유의확률 (p-value): {p_val:.4e}")
print(f"   - 모형 예측 정확도 (Concordance Index): {concordance_val:.4f}")
print(f"   - 모델 정보 손실률 (Partial AIC): {aic_val:.2f}")
print("-" * 65)
print("💡 [분석 결과 해석 가이드]")
print("   - 위험비(exp(coef))가 1보다 작고 p-value가 0.05보다 작다면,")
print("     '사업보고서 내 ESG 긍정 문장 개수가 많고 부정 문장 개수가 적을수록,")
print("      차년도 ESG 등급이 하락할 위험이 통계적으로 유의미하게 낮아진다'라고 결론을 기술합니다.")

### fasttext seed 사전 확장 상관분석/증분설명력 분석(지운)

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer

try:
    import fasttext
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"])
    import fasttext
    from huggingface_hub import hf_hub_download

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")
COMPANY_MASTER_PATH = Path("/content/drive/MyDrive/company_master.csv")

HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
HF_FASTTEXT_FILENAME = "model.bin"
FASTTEXT_TOP_K = 500
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.80, 0.90, 1.00]

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}
GRADE_BY_DIMENSION = {
    "E": "e_grade_num",
    "S": "s_grade_num",
    "G": "g_grade_num",
    "ESG": "esg_grade_num",
}


def normalize_term(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def token_key(term):
    return normalize_term(term).replace(" ", "_")


def threshold_label(theta):
    return f"{float(theta):.2f}".replace(".", "_")


def split_seed_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if term and term.lower() != "nan" and term not in seen:
            terms.append(term)
            seen.add(term)
    return terms


def build_seed_query_df(seed_df):
    rows = []
    for _, row in seed_df.iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue

        seed_term = normalize_term(row.get("seed_term", ""))
        for query_term in split_seed_terms(row):
            rows.append({
                "dimension": dimension,
                "seed_term": seed_term,
                "query_term": query_term,
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["dimension", "seed_term", "query_term"])
        .reset_index(drop=True)
    )


def build_seed_dictionary_df(seed_query_df):
    return (
        seed_query_df.rename(columns={"query_term": "candidate_term"})[
            ["dimension", "seed_term", "candidate_term"]
        ]
        .assign(dictionary="seed", threshold=np.nan, source="seed", similarity=1.0)
        .drop_duplicates(["dimension", "candidate_term"])
        .reset_index(drop=True)
    )


def safe_spearman(x, y):
    data = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 3 or data["x"].nunique() < 2 or data["y"].nunique() < 2:
        return np.nan, np.nan, len(data)

    rho, pvalue = spearmanr(data["x"], data["y"])
    return float(rho), float(pvalue), len(data)


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_query_df = build_seed_query_df(seed_source_df)
seed_dictionary_df = build_seed_dictionary_df(seed_query_df)

print("Seed query terms")
display(seed_query_df.groupby("dimension").size().rename("query_terms").reset_index())

model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
fasttext_model = fasttext.load_model(model_path)

raw_candidate_rows = []
for _, row in seed_query_df.iterrows():
    query_term = row["query_term"]
    try:
        neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
    except Exception as exc:
        print(f"Skipping {query_term!r}: {exc}")
        continue

    for similarity, candidate_term in neighbors:
        candidate_term = normalize_term(candidate_term)
        if not candidate_term or candidate_term == query_term:
            continue
        raw_candidate_rows.append({
            "dimension": row["dimension"],
            "seed_term": row["seed_term"],
            "query_term": query_term,
            "candidate_term": candidate_term,
            "similarity": float(similarity),
            "source": "fasttext_candidate",
        })

raw_fasttext_candidate_df = pd.DataFrame(raw_candidate_rows)
if raw_fasttext_candidate_df.empty:
    raise ValueError("No fastText candidates were generated. Check seed terms and model loading.")

raw_fasttext_candidate_df = raw_fasttext_candidate_df.drop_duplicates(
    ["dimension", "seed_term", "query_term", "candidate_term"]
).sort_values(["dimension", "seed_term", "query_term", "similarity"], ascending=[True, True, True, False])


def build_expanded_dictionary(theta):
    filtered = raw_fasttext_candidate_df[raw_fasttext_candidate_df["similarity"].ge(theta)].copy()
    expanded_rows = []
    for (dimension, candidate_term), group in filtered.groupby(["dimension", "candidate_term"], sort=False):
        best = group.sort_values("similarity", ascending=False).iloc[0]
        expanded_rows.append({
            "dictionary": f"fasttext_theta_{threshold_label(theta)}",
            "threshold": theta,
            "dimension": dimension,
            "seed_term": best["seed_term"],
            "candidate_term": candidate_term,
            "similarity": float(best["similarity"]),
            "source": "fasttext_candidate",
        })

    expanded_df = pd.DataFrame(expanded_rows)
    seed_rows = seed_dictionary_df.copy()
    seed_rows["dictionary"] = f"fasttext_theta_{threshold_label(theta)}"
    seed_rows["threshold"] = theta

    return (
        pd.concat([seed_rows, expanded_df], ignore_index=True)
        .drop_duplicates(["dimension", "candidate_term"], keep="first")
        .reset_index(drop=True)
    )


expanded_dictionary_by_threshold = {
    theta: build_expanded_dictionary(theta)
    for theta in THRESHOLDS
}

expanded_count_rows = []
for theta, dictionary_df in expanded_dictionary_by_threshold.items():
    for dimension in ["E", "S", "G"]:
        sub = dictionary_df[dictionary_df["dimension"].eq(dimension)]
        expanded_count_rows.append({
            "threshold": theta,
            "dimension": dimension,
            "total_terms": len(sub),
            "seed_terms": int(sub["source"].eq("seed").sum()),
            "fasttext_terms": int(sub["source"].eq("fasttext_candidate").sum()),
        })

print("FastText expanded term counts by theta")
display(pd.DataFrame(expanded_count_rows).pivot(index="threshold", columns="dimension", values="fasttext_terms"))

kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text):
    terms = []
    current = []

    for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
        form = normalize_term(token.form)
        if token.tag in NOUN_TAGS and len(form) > 1:
            terms.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                terms.append(normalize_term(" ".join(current)))
            current = []

    if len(current) >= 2:
        terms.append(normalize_term(" ".join(current)))

    return [term for term in terms if term]


appendix_kiwi_doc_df = corpus_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "document_norm",
]].copy()
appendix_kiwi_doc_df["kiwi_terms"] = appendix_kiwi_doc_df["document_norm"].apply(kiwi_noun_candidates)
appendix_kiwi_doc_df["kiwi_document"] = appendix_kiwi_doc_df["kiwi_terms"].apply(
    lambda terms: " ".join(token_key(term) for term in terms)
)
appendix_kiwi_doc_df["kiwi_term_count"] = appendix_kiwi_doc_df["kiwi_terms"].apply(len)

vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=1,
    norm=None,
)
appendix_tfidf_matrix = vectorizer.fit_transform(appendix_kiwi_doc_df["kiwi_document"])
appendix_vocab = vectorizer.vocabulary_


def prepare_terms(terms):
    prepared = []
    seen = set()
    for term in terms:
        term = normalize_term(term)
        key = token_key(term)
        if term and key in appendix_vocab and term not in seen:
            prepared.append((term, appendix_vocab[key]))
            seen.add(term)
    return prepared


def add_dictionary_scores(score_df, meta_rows, dictionary, threshold, dimension, terms, prefix):
    prepared_terms = prepare_terms(terms)
    term_set = {term for term, _ in prepared_terms}
    tfidf_cols = [col for _, col in prepared_terms]

    count_feature = f"{dimension}_{prefix}_count"
    tfidf_feature = f"{dimension}_{prefix}_tfidf"

    score_df[count_feature] = [
        sum(1 for term in doc_terms if term in term_set)
        for doc_terms in appendix_kiwi_doc_df["kiwi_terms"]
    ]
    score_df[tfidf_feature] = (
        np.asarray(appendix_tfidf_matrix[:, tfidf_cols].sum(axis=1)).ravel()
        if tfidf_cols else np.zeros(len(score_df), dtype=float)
    )

    meta_rows.append({
        "dictionary": dictionary,
        "threshold": threshold,
        "dimension": dimension,
        "prefix": prefix,
        "term_count": len(prepared_terms),
        "dictionary_term_count": len({normalize_term(term) for term in terms if normalize_term(term)}),
    })


appendix_score_df = appendix_kiwi_doc_df[[
    "stock_code", "company_name", "fiscal_year", "esg_year",
    "rcept_no", "total_word_count", "kiwi_term_count",
]].copy()
appendix_meta_rows = []

for dimension in ["E", "S", "G"]:
    seed_terms = seed_dictionary_df.loc[seed_dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
    add_dictionary_scores(appendix_score_df, appendix_meta_rows, "seed", np.nan, dimension, seed_terms, "seed")

for theta, dictionary_df in expanded_dictionary_by_threshold.items():
    dictionary = f"fasttext_theta_{threshold_label(theta)}"
    prefix = f"fasttext_{threshold_label(theta)}"
    for dimension in ["E", "S", "G"]:
        terms = dictionary_df.loc[dictionary_df["dimension"].eq(dimension), "candidate_term"].tolist()
        add_dictionary_scores(appendix_score_df, appendix_meta_rows, dictionary, theta, dimension, terms, prefix)

appendix_meta_df = pd.DataFrame(appendix_meta_rows)
for prefix in appendix_meta_df["prefix"].drop_duplicates():
    for score_type in ["count", "tfidf"]:
        appendix_score_df[f"ESG_{prefix}_{score_type}"] = sum(
            appendix_score_df[f"{dimension}_{prefix}_{score_type}"]
            for dimension in ["E", "S", "G"]
        )

company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
company_master["stock_code"] = company_master["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)
appendix_score_df["stock_code"] = appendix_score_df["stock_code"].astype("string").str.extract(r"(\d+)", expand=False).str.zfill(6)

for col in ["fiscal_year", "esg_year"]:
    company_master[col] = pd.to_numeric(company_master[col], errors="coerce").astype("Int64")
    appendix_score_df[col] = pd.to_numeric(appendix_score_df[col], errors="coerce").astype("Int64")

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    company_master[f"{col}_num"] = company_master[col].map(GRADE_MAP)

grade_cols = [
    "stock_code", "fiscal_year", "esg_year",
    "esg_grade_num", "e_grade_num", "s_grade_num", "g_grade_num",
]
appendix_analysis_df = appendix_score_df.merge(
    company_master[grade_cols].drop_duplicates(["stock_code", "fiscal_year", "esg_year"]),
    on=["stock_code", "fiscal_year", "esg_year"],
    how="left",
)

appendix_corr_rows = []
for _, meta in appendix_meta_df.iterrows():
    dimension = meta["dimension"]
    grade_col = GRADE_BY_DIMENSION[dimension]
    for score_type in ["count", "tfidf"]:
        feature = f"{dimension}_{meta['prefix']}_{score_type}"
        rho, pvalue, n = safe_spearman(appendix_analysis_df[feature], appendix_analysis_df[grade_col])
        appendix_corr_rows.append({
            "dictionary": meta["dictionary"],
            "threshold": meta["threshold"],
            "dimension": dimension,
            "score_type": score_type,
            "feature": feature,
            "grade_col": grade_col,
            "term_count": int(meta["term_count"]),
            "dictionary_term_count": int(meta["dictionary_term_count"]),
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

for _, group in appendix_meta_df.groupby(["dictionary", "threshold", "prefix"], dropna=False):
    dictionary = group["dictionary"].iloc[0]
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    term_count = int(group["term_count"].sum())
    dictionary_term_count = int(group["dictionary_term_count"].sum())
    for score_type in ["count", "tfidf"]:
        feature = f"ESG_{prefix}_{score_type}"
        rho, pvalue, n = safe_spearman(appendix_analysis_df[feature], appendix_analysis_df["esg_grade_num"])
        appendix_corr_rows.append({
            "dictionary": dictionary,
            "threshold": threshold,
            "dimension": "ESG",
            "score_type": score_type,
            "feature": feature,
            "grade_col": "esg_grade_num",
            "term_count": term_count,
            "dictionary_term_count": dictionary_term_count,
            "spearman_rho": rho,
            "p_value": pvalue,
            "n": n,
        })

appendix_fasttext_correlation_df = (
    pd.DataFrame(appendix_corr_rows)
    .sort_values(["dimension", "score_type", "threshold"], na_position="first")
    .reset_index(drop=True)
)

print("Appendix in-cell fastText expansion Spearman by theta: rho and p-value")
display(appendix_fasttext_correlation_df)

for score_type in ["count", "tfidf"]:
    expanded_result = appendix_fasttext_correlation_df[
        appendix_fasttext_correlation_df["dictionary"].ne("seed")
        & appendix_fasttext_correlation_df["score_type"].eq(score_type)
    ]

    print(f"Expanded fastText rho by theta - {score_type}")
    display(expanded_result.pivot_table(
        index="threshold",
        columns="dimension",
        values="spearman_rho",
        aggfunc="first",
    ))

    print(f"Expanded fastText p-value by theta - {score_type}")
    display(expanded_result.pivot_table(
        index="threshold",
        columns="dimension",
        values="p_value",
        aggfunc="first",
    ))

print("Seed baseline")
display(appendix_fasttext_correlation_df[appendix_fasttext_correlation_df["dictionary"].eq("seed")])

print("Observed expanded term counts in Kiwi vocabulary")
display(
    appendix_meta_df[appendix_meta_df["dictionary"].ne("seed")]
    .pivot_table(index="threshold", columns="dimension", values="term_count", aggfunc="first")
)

# Incremental explanatory power: does fastText added-only score improve over seed?
import statsmodels.api as sm


def appendix_zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def appendix_fit_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        return None, None, reg_df
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(appendix_zscore)
    X = sm.add_constant(X, has_constant="add")
    plain_model = sm.OLS(y, X).fit()
    hc3_model = sm.OLS(y, X).fit(cov_type="HC3")
    return plain_model, hc3_model, reg_df


fasttext_incremental_rows = []
seed_prefix = "seed"
for _, group in appendix_meta_df[appendix_meta_df["dictionary"].ne("seed")].groupby(["dictionary", "threshold", "prefix"], dropna=False):
    dictionary = group["dictionary"].iloc[0]
    threshold = group["threshold"].iloc[0]
    prefix = group["prefix"].iloc[0]
    added_term_count = int(group["term_count"].sum() - appendix_meta_df.loc[appendix_meta_df["prefix"].eq(seed_prefix), "term_count"].sum())
    added_dictionary_term_count = int(group["dictionary_term_count"].sum() - appendix_meta_df.loc[appendix_meta_df["prefix"].eq(seed_prefix), "dictionary_term_count"].sum())

    for score_type in ["count", "tfidf"]:
        seed_feature = f"ESG_{seed_prefix}_{score_type}"
        expanded_feature = f"ESG_{prefix}_{score_type}"
        added_feature = f"ESG_{prefix}_added_only_{score_type}"
        if seed_feature not in appendix_analysis_df.columns or expanded_feature not in appendix_analysis_df.columns:
            continue

        # fastText expanded dictionaries include seed rows, so subtract seed to isolate added terms.
        appendix_analysis_df[added_feature] = appendix_analysis_df[expanded_feature] - appendix_analysis_df[seed_feature]

        base_plain, base_hc3, base_df = appendix_fit_ols(
            appendix_analysis_df,
            "esg_grade_num",
            [seed_feature, "total_word_count"],
        )
        full_plain, full_hc3, full_df = appendix_fit_ols(
            appendix_analysis_df,
            "esg_grade_num",
            [seed_feature, added_feature, "total_word_count"],
        )
        if base_plain is None or full_plain is None:
            continue

        f_stat, f_pvalue, df_diff = full_plain.compare_f_test(base_plain)
        fasttext_incremental_rows.append({
            "dictionary": dictionary,
            "threshold": threshold,
            "score_type": score_type,
            "seed_feature": seed_feature,
            "added_feature": added_feature,
            "n": int(full_plain.nobs),
            "added_term_count": added_term_count,
            "added_dictionary_term_count": added_dictionary_term_count,
            "base_r2": float(base_plain.rsquared),
            "full_r2": float(full_plain.rsquared),
            "delta_r2": float(full_plain.rsquared - base_plain.rsquared),
            "base_adj_r2": float(base_plain.rsquared_adj),
            "full_adj_r2": float(full_plain.rsquared_adj),
            "delta_adj_r2": float(full_plain.rsquared_adj - base_plain.rsquared_adj),
            "base_aic": float(base_plain.aic),
            "full_aic": float(full_plain.aic),
            "delta_aic_full_minus_base": float(full_plain.aic - base_plain.aic),
            "base_bic": float(base_plain.bic),
            "full_bic": float(full_plain.bic),
            "delta_bic_full_minus_base": float(full_plain.bic - base_plain.bic),
            "partial_f": float(f_stat),
            "partial_f_p_value": float(f_pvalue),
            "df_diff": float(df_diff),
            "added_coef_hc3": float(full_hc3.params[added_feature]),
            "added_p_value_hc3": float(full_hc3.pvalues[added_feature]),
        })

appendix_fasttext_incremental_ols_df = (
    pd.DataFrame(fasttext_incremental_rows)
    .sort_values(["score_type", "delta_adj_r2"], ascending=[True, False])
    .reset_index(drop=True)
)

print("FastText added-only incremental OLS over seed baseline")
display(appendix_fasttext_incremental_ols_df)

if not appendix_fasttext_incremental_ols_df.empty:
    print("Best FastText threshold by adjusted R2 gain")
    display(
        appendix_fasttext_incremental_ols_df
        .sort_values("delta_adj_r2", ascending=False)
        .head(10)
    )



### 확장 사전 csv 저장

In [ ]:
# Save only the selected expanded dictionary candidates to Google Drive.
# This cell recalculates only fastText theta=0.70 and Kiwi embedding threshold=0.80,
# so it does not require running the appendix threshold-sweep cells.
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import fasttext
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "fasttext-wheel", "huggingface_hub"])
    import fasttext
    from huggingface_hub import hf_hub_download

try:
    from kiwipiepy import Kiwi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy"])
    from kiwipiepy import Kiwi

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm

SEED_DICTIONARY_PATH = Path("/content/drive/MyDrive/seed_dictionary.csv")
OUTPUT_DIR = Path("/content/drive/MyDrive")

BEST_FASTTEXT_THETA = 0.70
BEST_KIWI_THRESHOLD = 0.80

HF_FASTTEXT_REPO_ID = "facebook/fasttext-ko-vectors"
HF_FASTTEXT_FILENAME = "model.bin"
FASTTEXT_TOP_K = 500

KIWI_MODEL_NAME = "dragonkue/BGE-m3-ko"
KIWI_MIN_TERM_FREQ = 3
KIWI_MIN_DOC_FREQ = 2
KIWI_MAX_CANDIDATES = 30000
KIWI_BATCH_SIZE = 64


def normalize_term(value):
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text.strip(" \t\r\n\"'`.,;:()[]{}<>")


def threshold_label(theta):
    return f"{float(theta):.2f}".replace(".", "_")


def split_seed_terms(row):
    values = [row.get("seed_term", "")]
    pattern = row.get("pattern", "")
    if pd.notna(pattern):
        values.extend(str(pattern).split("|"))

    terms = []
    seen = set()
    for value in values:
        term = normalize_term(value)
        if term and term.lower() != "nan" and term not in seen:
            terms.append(term)
            seen.add(term)
    return terms


def build_seed_query_df(seed_df):
    rows = []
    for _, row in seed_df.iterrows():
        dimension = normalize_term(row.get("dimension", ""))
        if dimension not in {"E", "S", "G"}:
            continue

        seed_term = normalize_term(row.get("seed_term", ""))
        for query_term in split_seed_terms(row):
            rows.append({
                "dimension": dimension,
                "seed_term": seed_term,
                "query_term": query_term,
                "query_text": query_term.replace("_", " "),
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates(["dimension", "query_term"])
        .reset_index(drop=True)
    )


def build_fasttext_dictionary(seed_query_df, theta):
    print(f"Downloading/loading fastText model from {HF_FASTTEXT_REPO_ID}/{HF_FASTTEXT_FILENAME}...")
    model_path = hf_hub_download(repo_id=HF_FASTTEXT_REPO_ID, filename=HF_FASTTEXT_FILENAME)
    print(f"Loading fastText model: {model_path}")
    fasttext_model = fasttext.load_model(model_path)
    print("fastText model loaded. Collecting nearest neighbors...")

    candidate_rows = []
    seed_iter = tqdm(
        seed_query_df.iterrows(),
        total=len(seed_query_df),
        desc=f"fastText theta>={theta:.2f}",
        unit="seed",
    )
    for _, row in seed_iter:
        query_term = row["query_term"]
        try:
            neighbors = fasttext_model.get_nearest_neighbors(query_term, k=FASTTEXT_TOP_K)
        except Exception:
            continue

        for similarity, candidate in neighbors:
            candidate_term = normalize_term(candidate.replace("_", " "))
            if not candidate_term or similarity < theta:
                continue
            candidate_rows.append({
                "dictionary": f"fasttext_theta_{threshold_label(theta)}",
                "threshold": theta,
                "dimension": row["dimension"],
                "seed_term": row["seed_term"],
                "query_term": query_term,
                "candidate_term": candidate_term,
                "source": "fasttext_neighbor",
                "similarity": float(similarity),
            })

    seed_rows = (
        seed_query_df.rename(columns={"query_term": "candidate_term"})[
            ["dimension", "seed_term", "candidate_term"]
        ]
        .drop_duplicates(["dimension", "candidate_term"])
        .copy()
    )
    seed_rows["dictionary"] = f"fasttext_theta_{threshold_label(theta)}"
    seed_rows["threshold"] = theta
    seed_rows["query_term"] = seed_rows["candidate_term"]
    seed_rows["source"] = "seed"
    seed_rows["similarity"] = 1.0

    print(f"fastText candidates kept at theta>={theta:.2f}: {len(candidate_rows):,}")
    candidate_df = pd.DataFrame(candidate_rows)
    dictionary_df = pd.concat([seed_rows, candidate_df], ignore_index=True, sort=False)
    return (
        dictionary_df
        .drop_duplicates(["dimension", "candidate_term"], keep="first")
        .sort_values(["dimension", "source", "similarity", "candidate_term"], ascending=[True, False, False, True])
        .reset_index(drop=True)
    )


kiwi = Kiwi()
NOUN_TAGS = {"NNG", "NNP", "SL"}


def kiwi_noun_candidates(text):
    terms = []
    current = []
    for token in kiwi.tokenize("" if pd.isna(text) else str(text)):
        form = normalize_term(token.form)
        if token.tag in NOUN_TAGS and len(form) > 1:
            terms.append(form)
            current.append(form)
        else:
            if len(current) >= 2:
                terms.append(normalize_term(" ".join(current)))
            current = []

    if len(current) >= 2:
        terms.append(normalize_term(" ".join(current)))

    return [term for term in terms if term]


def encode_texts(model, texts, batch_size):
    return model.encode(
        list(texts),
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


def build_kiwi_dictionary(seed_query_df, threshold):
    if "corpus_df" not in globals():
        raise RuntimeError("corpus_df is required. Run the corpus-building cells before this save cell.")
    if "document_norm" not in corpus_df.columns:
        raise RuntimeError("corpus_df must contain a document_norm column.")

    doc_terms = corpus_df["document_norm"].apply(kiwi_noun_candidates)
    term_frequency = Counter(term for terms in doc_terms for term in terms)
    doc_frequency = Counter(term for terms in doc_terms for term in set(terms))

    seed_terms = set(seed_query_df["query_term"])
    candidate_terms = [
        term for term, freq in term_frequency.most_common()
        if freq >= KIWI_MIN_TERM_FREQ
        and doc_frequency[term] >= KIWI_MIN_DOC_FREQ
        and term not in seed_terms
    ][:KIWI_MAX_CANDIDATES]

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = KIWI_BATCH_SIZE if device == "cuda" else min(16, KIWI_BATCH_SIZE)
    model = SentenceTransformer(KIWI_MODEL_NAME, device=device)

    seed_embeddings = encode_texts(model, seed_query_df["query_text"], batch_size)
    dimension_indices = {
        dimension: list(indices)
        for dimension, indices in seed_query_df.groupby("dimension").groups.items()
    }

    candidate_rows = []
    for start in range(0, len(candidate_terms), batch_size):
        batch_terms = candidate_terms[start:start + batch_size]
        batch_embeddings = encode_texts(model, batch_terms, batch_size)
        similarity_matrix = batch_embeddings @ seed_embeddings.T

        for row_i, candidate_term in enumerate(batch_terms):
            similarities = similarity_matrix[row_i]
            best_i = int(np.argmax(similarities))
            best_seed = seed_query_df.iloc[best_i]
            best_dimension = best_seed["dimension"]
            best_dimension_score = float(similarities[best_i])

            if best_dimension_score < threshold:
                continue

            dimension_scores = {
                dimension: float(np.max(similarities[indices]))
                for dimension, indices in dimension_indices.items()
            }
            ordered_scores = sorted(dimension_scores.values(), reverse=True)
            second_dimension_score = ordered_scores[1] if len(ordered_scores) > 1 else np.nan

            candidate_rows.append({
                "dictionary": "kiwi_embedding_expanded",
                "threshold": threshold,
                "dimension": best_dimension,
                "candidate_term": candidate_term,
                "source": "kiwi_embedding_candidate",
                "seed_term": best_seed["seed_term"],
                "best_query_term": best_seed["query_term"],
                "best_dimension_score": best_dimension_score,
                "second_dimension_score": second_dimension_score,
                "dimension_margin": best_dimension_score - second_dimension_score if not np.isnan(second_dimension_score) else np.nan,
                "term_frequency": int(term_frequency[candidate_term]),
                "doc_frequency": int(doc_frequency[candidate_term]),
            })

    kiwi_seed_rows = (
        seed_query_df.rename(columns={"query_term": "candidate_term"})[
            ["dimension", "seed_term", "candidate_term"]
        ]
        .drop_duplicates(["dimension", "candidate_term"])
        .copy()
    )
    kiwi_seed_rows["dictionary"] = "kiwi_embedding_expanded"
    kiwi_seed_rows["threshold"] = threshold
    kiwi_seed_rows["source"] = "seed"
    kiwi_seed_rows["best_dimension_score"] = 1.0
    kiwi_seed_rows["second_dimension_score"] = pd.NA
    kiwi_seed_rows["dimension_margin"] = pd.NA
    kiwi_seed_rows["best_query_term"] = kiwi_seed_rows["candidate_term"]
    kiwi_seed_rows["term_frequency"] = pd.NA
    kiwi_seed_rows["doc_frequency"] = pd.NA

    kiwi_columns = [
        "dictionary", "threshold", "dimension", "candidate_term", "source",
        "seed_term", "best_query_term", "best_dimension_score",
        "second_dimension_score", "dimension_margin", "term_frequency", "doc_frequency",
    ]

    return (
        pd.concat([kiwi_seed_rows, pd.DataFrame(candidate_rows)], ignore_index=True, sort=False)
        .drop_duplicates(["dimension", "candidate_term"], keep="first")
        .reindex(columns=kiwi_columns)
        .sort_values(["dimension", "source", "best_dimension_score", "candidate_term"], ascending=[True, False, False, True])
        .reset_index(drop=True)
    )


seed_source_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")
seed_query_df = build_seed_query_df(seed_source_df)

# Candidate 1: fastText expansion, selected by the appendix incremental OLS result.
best_fasttext_dictionary_df = build_fasttext_dictionary(seed_query_df, BEST_FASTTEXT_THETA)
best_fasttext_dictionary_df["selected_by"] = "best_incremental_ols_candidate"
best_fasttext_dictionary_df["selection_note"] = "fastText theta=0.70 had the strongest incremental explanatory power over the seed baseline in the appendix"

fasttext_output_path = OUTPUT_DIR / "candidate_fasttext_theta_0_70_dictionary.csv"
best_fasttext_dictionary_df.to_csv(fasttext_output_path, index=False, encoding="utf-8-sig")

# Candidate 2: Kiwi embedding expansion, selected by the appendix incremental OLS result.
best_kiwi_dictionary_df = build_kiwi_dictionary(seed_query_df, BEST_KIWI_THRESHOLD)
best_kiwi_dictionary_df["selected_by"] = "best_incremental_ols_candidate"
best_kiwi_dictionary_df["selection_note"] = "Kiwi embedding threshold=0.80 had the strongest incremental explanatory power over the seed baseline in the appendix"

kiwi_output_path = OUTPUT_DIR / "candidate_kiwi_embedding_threshold_0_80_dictionary.csv"
best_kiwi_dictionary_df.to_csv(kiwi_output_path, index=False, encoding="utf-8-sig")

print("Saved candidate expanded dictionaries:")
print(f"- fastText theta={BEST_FASTTEXT_THETA:.2f}: {fasttext_output_path}")
print(f"- Kiwi embedding threshold={BEST_KIWI_THRESHOLD:.2f}: {kiwi_output_path}")

print("\nfastText terms by dimension/source")
display(best_fasttext_dictionary_df.groupby(["dimension", "source"]).size().rename("terms").reset_index())

print("Kiwi embedding terms by dimension/source")
display(best_kiwi_dictionary_df.groupby(["dimension", "source"]).size().rename("terms").reset_index())
